# Predict effect of variants

Here we provide examples of scoring genetic sequences with variants.<br>
You can predict the effect of a single variant or compose dataset to score a multidute of variants. <br>
**Note:** to do the latter, you must provide tab-delimited table with the following columns: <br>
[chrom, pos, ref, alt, variant_id]

## Imports

In [1]:
from os.path import exists
from io import StringIO
from functools import reduce
from IPython.display import display
from tqdm import tqdm_notebook, tqdm

import numpy as np
from numpy.typing import ArrayLike
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from genome_tools.genomic_interval import GenomicInterval
from genome_tools.data.extractors import FastaExtractor
from genome_tools.data.anndata import read_zarr_backed
from vinson.datasets.sequence import CartesianInferenceDataset
from vinson.utils.sequence_utils import one_hot_encode
from vinson.utils.inference import InferenceDatasetKwargs, score_variants_with_dhs_model, assemble_batch, score_bed_with_dhs_model
from vinson.utils.helpers import replace_at
from vinson.from_config import read_configs, dhs_model_from_config
from vinson.postprocessing.interpretation import ModelWrapper

# from genome_tools.plotting.ideogram import read_ideogram
# from genome_tools.plotting.modular_plot import IntervalPlotter
# from genome_tools.plotting.modular_plot.plot_components.prediction import PredictedSignalComponent
# from genome_tools.plotting.modular_plot.plot_components.basic import TrackComponent, IdeogramComponent, GencodeComponent


findfont: Font family ['Arial'] not found. Falling back to DejaVu Sans.


In [2]:
# we provide with example variant_df to score with model

str_variants = """chrom\tpos\tref\talt
chr10\t11154768\tC\tG
chr3\t168285216\tC\tG
chr4\t153440582\tG\tA
chr8\t53311365\tA\tG
chr18\t5425302\tC\tT
chr4\t100185159\tT\tC
chr17\t82969252\tC\tT
chr16\t77722716\tG\tA"""

In [3]:
device = 'cuda:1'


fasta_file = '/mnt/calc/sbushuev/genomes/hg38/hg38.fa' # genome
genotype_file = '/mnt/calc/sbushuev/dhs_project/JUN29/all_variants_stats.trimmed_cols.bed.gz'
config_path= '/mnt/calc/homes/sbushuev/dhs_project/configs/FinalVersion/MAR06_8mln_long.yaml' # path to model's config .yaml file used to train model
checkpoint_path = '/mnt/calc/homes/sbushuev/dhs_project/checkpoints/9epochs_8mln_JUN29_best_epoch=8.ckpt' # path to .ckpt file
embed_path = '/mnt/calc/homes/sbushuev/dhs_project/embeds/data.all.tsv'
embed_meta_path = '/mnt/calc/homes/sbushuev/dhs_project/embeds/meta.all.tsv'

## Inject variant into single sequence

Select sample's embedding, load model, present genetic variant and predict its effect 

In [4]:
## set up model
config = read_configs(config_path)
model = dhs_model_from_config(config, checkpoint_path = checkpoint_path).eval().to(device) #L.LightningModule

embed_df = pd.read_table(embed_path, index_col=0)
embed_meta_df = pd.read_table(embed_meta_path, index_col=0)

/mnt/calc/sbushuev/.conda/envs/dhs_env/lib/python3.12/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [5]:
## your dna sequence
half_seqlen = 1344//2
interval = GenomicInterval.from_ucsc('chr2:60495243-60495243').center.widen(half_seqlen) # select genomic loci
fasta_extr = FastaExtractor(fasta_file)
dna_seq_ref = fasta_extr[interval].upper() # extract sequence

##select sample and its embedding
chosen_term = 'HUDEP-2'
embed_id = embed_meta_df.loc[embed_meta_df.core_ontology_term == chosen_term].index.values[0] # chose one sample
embed = embed_df.loc[embed_id].values

## SNP: C>A
mid = half_seqlen
dna_seq_alt = replace_at(dna_seq_ref, mid , 'A') # inject SNP into the middle of sequence 
## convert to torch.Tensor and reshape to mimic batch
ohe_seq_ref = torch.as_tensor(one_hot_encode(dna_seq_ref), device=device, dtype=torch.float32)[None,:, :]
ohe_seq_alt = torch.as_tensor(one_hot_encode(dna_seq_alt), device=device, dtype=torch.float32)[None,:, :]

embed = torch.as_tensor(embed, device=device, dtype=torch.float32)[None,:]

## get your prediction
pred_ref, pred_alt = model(ohe_seq_ref, embed), model(ohe_seq_alt, embed)
print('Predictions for alt and ref alleles:',pred_alt.item(), pred_ref.item())

Predictions for alt and ref alleles: 3.089637279510498 3.4820778369903564


/mnt/calc/sbushuev/.conda/envs/dhs_env/lib/python3.12/site-packages/torch/nn/modules/conv.py:304: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /opt/conda/conda-bld/pytorch_1724789220573/work/aten/src/ATen/native/Convolution.cpp:1031.)
  return F.conv1d(input, weight, bias, self.stride,


## Predict effect of variants <br>in the context of different cell-types

To predict effects of multiple genetic variants we suggest to use test-time augmentation with shifts. <br>

In [6]:
config = read_configs(config_path)
embed_df = pd.read_table(embed_path, index_col=0)
embed_meta = pd.read_table(embed_meta_path, index_col=0)
variants = pd.read_table(StringIO(str_variants))

model = dhs_model_from_config(config, checkpoint_path = checkpoint_path).eval().to(device)

## Select samples, shifts to score
shifts = [-10, 0, 10]
selected_samples = ['AG80837', 'AG80853']
embed_df = embed_df.loc[selected_samples]

## establish config for inference
dataset_kwargs = InferenceDatasetKwargs(embed_df=embed_df,
                                        variants_df=variants,
                                        shifts=shifts,
                                        fasta_file=fasta_file,
                                       )

/mnt/calc/sbushuev/.conda/envs/dhs_env/lib/python3.12/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [7]:
## set up kwargs for dataloader
dataloader_kwargs = dict(batch_size=256, 
                        num_workers=16, 
                        pin_memory=True, 
                        shuffle=False)

In [8]:
out_df, (alt_scores, ref_scores) = score_variants_with_dhs_model(dataset_kwargs, model,
                              dataloader_kwargs=dataloader_kwargs,
                              device=device)
out_df.head()

  0%|          | 0/1 [00:00<?, ?it/s]

,chrom,pos,ref,alt,shift,embed_id,var_id
0,chr10,11154768,C,G,-10,AG80837,chr10_11154768_C_G
1,chr3,168285216,C,G,-10,AG80837,chr3_168285216_C_G
2,chr4,153440582,G,A,-10,AG80837,chr4_153440582_G_A
3,chr8,53311365,A,G,-10,AG80837,chr8_53311365_A_G
4,chr18,5425302,C,T,-10,AG80837,chr18_5425302_C_T


Raw output table might be very large for large range of shifts and multidute of chosen samples, be carefull.

### Aggregate your predictions however you want: across samples, across shifts, etc

In [ ]:
out_df['vinson.ref_score'] = np.log(ref_scores)
out_df['vinson.alt_score'] = np.log(alt_scores)
aggr_df = out_df.groupby(['chrom', 'pos', 'ref', 'alt','embed_id' ,'var_id'], observed=True ).agg(
    {'vinson.ref_score':'mean',
     'vinson.alt_score':'mean'}
).reset_index()
aggr_df['vinson.ref_score'] = np.exp(aggr_df['vinson.ref_score'])
aggr_df['vinson.alt_score'] = np.exp(aggr_df['vinson.alt_score'])

aggr_df.head()

,chrom,pos,ref,alt,embed_id,var_id,vinson.ref_score,vinson.alt_score
0,chr10,11154768,C,G,AG80837,chr10_11154768_C_G,0.020111,0.021039
1,chr10,11154768,C,G,AG80853,chr10_11154768_C_G,0.026137,0.027271
2,chr16,77722716,G,A,AG80837,chr16_77722716_G_A,1.239882,1.324736
3,chr16,77722716,G,A,AG80853,chr16_77722716_G_A,0.741653,0.814745
4,chr17,82969252,C,T,AG80837,chr17_82969252_C_T,0.429942,0.456354


## Inject genotypes

You can predict effects of all genetic variants in genomic loci.  <br>
To enable it you must provide your **tab-delimited genotype_file** without header with following columns:<br>
chromosome, pos, pos+1, ref, alt, indiv_id of individual in which variant was detected and allele dosage. <br>
**Note:** you with genotype file you must provide **tab-delimited meta table for embeddings (embed_meta_df....)**<br> 
that must has first column with sample_id for which you have embeddings with motif activities <br>and inviv_id column that matches to column in genotype file.

In [9]:
config = read_configs(config_path)
embed_df = pd.read_table(embed_path, index_col=0)
embed_meta = pd.read_table(embed_meta_path, index_col=0)
variants = pd.read_table(StringIO(str_variants))

model = dhs_model_from_config(config, checkpoint_path = checkpoint_path).eval().to(device)

## Select samples, shifts to score
shifts = [-10, 0, 10]
selected_samples = ['AG80837', 'AG80853']
embed_df = embed_df.loc[selected_samples]

## establish config for inference
dataset_kwargs = InferenceDatasetKwargs(embed_df=embed_df,
                                        variants_df=variants,
                                        shifts=shifts,
                                        fasta_file=fasta_file,
                                        genotype_file=genotype_file,
                                        embed_meta_df=embed_meta_df
                                       )

/mnt/calc/sbushuev/.conda/envs/dhs_env/lib/python3.12/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [10]:
## set up kwargs for dataloader
dataloader_kwargs = dict(batch_size=256, 
                        num_workers=16, 
                        pin_memory=True, 
                        shuffle=False)

In [11]:
out_df, (alt_scores, ref_scores) = score_variants_with_dhs_model(dataset_kwargs, model,
                              dataloader_kwargs=dataloader_kwargs,
                              device=device)
out_df.head()

  0%|          | 0/1 [00:00<?, ?it/s]

,chrom,pos,ref,alt,shift,embed_id,var_id
0,chr10,11154768,C,G,-10,AG80837,chr10_11154768_C_G
1,chr3,168285216,C,G,-10,AG80837,chr3_168285216_C_G
2,chr4,153440582,G,A,-10,AG80837,chr4_153440582_G_A
3,chr8,53311365,A,G,-10,AG80837,chr8_53311365_A_G
4,chr18,5425302,C,T,-10,AG80837,chr18_5425302_C_T
